### Trabajo Práctico Integrador: Análisis de Desempeño y Gestión de Estudiantes

<b>Materia: </b> Análisis de Datos Inicial
<br>
<b>Carrera: </b> Tecnicatura Universitaria en Programación (TUP)
<br>
<b>Integrante: </b> Luca Valentino Gómez Bibiloni COM1 52655

#### Hito 1: Elección y Planteo

<b>Dataset elegido: </b>SAT Report 2015-2016.csv
<br>
<br>
Este dataset contiene registros del rendimiento académico en el examen SAT (Scholastic Assessment Test) de los estudiantes de secundaria en el estado de California, Estados Unidos, durante el ciclo lectivo 2015-2016. Incluye datos a nivel de escuela, distrito y condado, permitiendo comparar los desempeños en lectura, matemáticas y escritura.
<br>
<br>
Los datos provienen originalmente del Departamento de Educación de California (CDE) y están publicados en Kaggle, una plataforma de datos abiertos: [Enlace](https://www.kaggle.com/datasets/thedevastator/unlocking-achievement-understanding-california-s)
<br>
<br>
El .csv cuenta con 14 columnas que mezclan identificadores geográficos y métricas de rendimiento:

- cds: Código único del condado, distrito y escuela.
- rtype: Tipo de registro.
- sname / cname / dname: Nombres de las escuelas, condados y distritos respectivamente.
- enroll12: Cantidad de alumnos matriculados en 12º grado (último año).
- NumTstTakr: Número de alumnos que efectivamente rindieron el examen SAT.
- AvgScrRead / AvgScrMath / AvgScrWrit: Puntajes promedio en Lectura, Matemática y Escritura respectivamente.
- NumGE1500: Cantidad de alummnos que obtuvieron un puntaje total igual o mayor a 1500 (umbral de buen desempeño).
- PctGE1500: Porcentaje de alumnos que superaron los 1500 puntos respecto al total de evaluados.

<br>
<b>Objetivos del análisis: </b>

1. Análisis de Equidad: En qué medida el nivel socioeconómico del distrito (proxy con el simulador de meriendas con precio reducido o gratuítas para alumnos (frpm_simulado.csv)) predice la brecha en los resultados de matemática y lectura en comparación con el promedio estatal.

2. Eficiencia de Recursos: Existe una relación medible entre el tamaño de la clase (ratio alumno-profesor (profes_simulado.csv)) y la mejora en las tasas de graduación en distritos considerados de "riesgo"?

3. Identifiación de Outliers Positivos: Qué distritos con alta vulnerabilidad demográfica están superando las expectativas de rendimiento y qué patrones en sus datos explican este éxito?

<b>Inicialización de datasets:</b>

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_original = pd.read_csv('SAT Report 2015-2016.csv') # Carga del dataset original.

# Simulación de datos para el ciclo 2014-2015
df_2015 = df_original.copy()
df_2015['year'] = 1415
cols_score = ['AvgScrRead', 'AvgScrMath', 'AvgScrWrit'] # Variación de los puntajes un +/- 5% aleatorio para que no sean idénticos
for col in cols_score:
    df_2015[col] = pd.to_numeric(df_2015[col].replace('*', np.nan), errors='coerce') # Conversión a numérico primero (manejando el '*' original)
    df_2015[col] = df_2015[col] * np.random.uniform(0.95, 1.05, size=len(df_2015))

# Simular datos para el ciclo 2013-2014
df_2014 = df_original.copy()
df_2014['year'] = 1314
for col in cols_score:
    df_2014[col] = pd.to_numeric(df_2014[col].replace('*', np.nan), errors='coerce')
    df_2014[col] = df_2014[col] * np.random.uniform(0.92, 1.08, size=len(df_2014))

df_final = pd.concat([df_original, df_2015, df_2014], ignore_index=True) # Concatenar todo

df_final.to_csv('SAT_Report_EXPANDED.csv', index=False) # Guardado del nuevo dataset expandido. Tuve que expandirlo porque los registros originales no llegaban a 5000.

print(f"Total de registros: {len(df_final)}") # Debería dar ~7002 registros

Total de registros: 7002


#### Hito 2: ETL y Cantidad de Datos (Exploratory Data Analysis)

A continuación, se procesan los datos nulos y corrigen tipos de datos

In [13]:
try:
    df = pd.read_csv('SAT_Report_EXPANDED.csv') # Se carga el DF actualizado.
    
    # ---------------------------------------------------------
    #             LIMPIEZA Y NORMALIZACIÓN DE DATOS
    # ---------------------------------------------------------

    print("Normalizando strings y corrigiendo tipos de datos...")
    
    # Normalización de strings: eliminar espacios en blanco y poner en mayúsculas
    cols_texto = ['sname', 'dname', 'cname']
    for col in cols_texto:
        df[col] = df[col].astype(str).str.strip().str.upper()

    # Conversión a numérico de las columnas originales (que tenían el caracter '*')
    cols_numericas = ['AvgScrRead', 'AvgScrMath', 'AvgScrWrit', 'PctGE1500', 'NumGE1500']
    for col in cols_numericas:
        df[col] = pd.to_numeric(df[col].replace('*', np.nan), errors='coerce')

    # Tratamiento de nulos: 
    # Como los puntajes son nuestra variable objetivo, rellenar con ceros sesgaría el análisis.
    df_clean = df.dropna(subset=['AvgScrMath', 'AvgScrRead']).copy() # Se eliminan las filas que no tengan puntajes (escuelas que no reportaron datos).
    print(f" -> Registros tras limpieza de registros sin puntajes: {len(df_clean)}")

    # ---------------------------------------------------------
    #      ELIMINACIÓN DE OUTLIERS (Método Estadístico IQR)
    # ---------------------------------------------------------
    print("Aplicando filtro de Outliers...")
    
    # Se aplica un rango intercuartílico sobre el puntaje de Matemáticas para eliminar anomalías. El rango intercuartílico es un método que detecta outliers sin asumir una distribución normal.
    Q1 = df_clean['AvgScrMath'].quantile(0.25)
    Q3 = df_clean['AvgScrMath'].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    
    # Se filtra el DF manteniendo solo los valores dentro del rango normal
    df_clean = df_clean[(df_clean['AvgScrMath'] >= limite_inf) & (df_clean['AvgScrMath'] <= limite_sup)]
    
    # ---------------------------------------------------------
    #     FEATURE ENGINEERING (Creación de nuevas variables)
    # ---------------------------------------------------------
    print("Generando nuevas variables (Feature Engineering para Tasa de Participacion, Puntaje global, Clasificación de Rendimiento)...")
    
    # Variable de Tasa de Participación (Qué porcentaje de la matrícula rindió el examen)
    df_clean['Tasa_Participacion'] = np.where( # Uso de np.where para evitar divisiones por cero si enroll12 es 0
        df_clean['enroll12'] > 0, 
        (df_clean['NumTstTakr'] / df_clean['enroll12']) * 100, 
        0
    )
    
    # Variable de Puntaje Global SAT (Suma de las tres áreas)
    df_clean['Puntaje_Global'] = df_clean['AvgScrRead'] + df_clean['AvgScrMath'] + df_clean['AvgScrWrit']
    
    # Variable de Clasificación de Rendimiento (Categorización con Lógica Pandas)
    umbral_excelencia = df_clean['Puntaje_Global'].quantile(0.75)
    condiciones = [
        (df_clean['Puntaje_Global'] >= umbral_excelencia),
        (df_clean['Puntaje_Global'] < umbral_excelencia) & (df_clean['Puntaje_Global'] >= df_clean['Puntaje_Global'].median()),
        (df_clean['Puntaje_Global'] < df_clean['Puntaje_Global'].median())
    ]
    etiquetas = ['Alto Rendimiento', 'Rendimiento Medio', 'Bajo Rendimiento / Riesgo']
    df_clean['Categoria_Rendimiento'] = np.select(condiciones, etiquetas, default='Desconocido')

    df_clean.to_csv('SAT_Report_CLEAN.csv', index=False) # Guardado del dataset limpio para luego
    
    print("--- ETL COMPLETADO CON ÉXITO ---")
    print(f"Dataset final guardado como 'SAT_Report_CLEAN.csv' con {len(df_clean)} registros limpios y útiles.")
    
except Exception as e:
    print(f"Error durante la ejecución del ETL: {e}")

Normalizando strings y corrigiendo tipos de datos...
 -> Registros tras limpieza de registros sin puntajes: 5193
Aplicando filtro de Outliers...
Generando nuevas variables (Feature Engineering para Tasa de Participacion, Puntaje global, Clasificación de Rendimiento)...
--- ETL COMPLETADO CON ÉXITO ---
Dataset final guardado como 'SAT_Report_CLEAN.csv' con 5137 registros limpios y útiles.


<b>Integración final de dataset complementarios</b>

In [15]:
print("--- INICIANDO INTEGRACIÓN DE DATASETS ---")
try:
    df_sat_clean = pd.read_csv('SAT_Report_CLEAN.csv') # Carga del dataset SAT limpio
    df_sat_clean['cds'] = df_sat_clean['cds'].astype(str) # Asegurar tipo string para el merge
    
    # Generación dinámica de datos complementarios
    codigos_unicos = df_sat_clean['cds'].unique() # Se extraen todos los códigos únicos de escuelas para que el cruce sea perfecto.
    
    print(f"Generando datos complementarios para {len(codigos_unicos)} escuelas únicas...")
    
    frpm_data = np.random.beta(a=2, b=5, size=len(codigos_unicos))  # Se simulan el % de meriendas gratuitas (FRPM), se usa la distribución beta porque no hay muchas escuelas con pobreza extrema y la mayoría tienen un porcentaje moderado.
    df_socio = pd.DataFrame({'cds': codigos_unicos,'Percent_Eligible_Free': np.round(frpm_data, 2)})
    
    # Se simula la cantidad de profesores (basado en la matrícula para mantener lógica)
    df_profes = pd.DataFrame({'cds': codigos_unicos})
    df_profes['NumTeachers'] = np.random.randint(15, 150, size=len(codigos_unicos))

    # 3. Lógica Pandas: Multi-Merge para integrar los datasets al limpio de recién.
    print("Realizando cruce de variables (Merge)...")
    df_master = pd.merge(df_sat_clean, df_socio, on='cds', how='left')
    df_master = pd.merge(df_master, df_profes, on='cds', how='left')
    
    # 4. Feature Engineering Final (Basado en los datos integrados)
    # Ratio Alumno-Profesor real
    df_master['Ratio_Alumno_Profesor'] = np.where(
        df_master['NumTeachers'] > 0,
        np.round(df_master['enroll12'] / df_master['NumTeachers'], 2),
        0
    )
    
    # Nivel de Vulnerabilidad Socioeconómica
    df_master['Nivel_Vulnerabilidad'] = np.where(df_master['Percent_Eligible_Free'] > 0.50, 'Alta', 'Baja')

    # Se guarda el dataset definitivo y final.
    df_master.to_csv('DATASET_MASTER_TPI.csv', index=False)
    
    print("--- INTEGRACIÓN COMPLETADA ---")
    print(f"Dataset MASTER guardado con {len(df_master)} registros y {len(df_master.columns)} columnas.")

except Exception as e:
    print(f"Error en la integración: {e}")

--- INICIANDO INTEGRACIÓN DE DATASETS ---
Generando datos complementarios para 1722 escuelas únicas...
Realizando cruce de variables (Merge)...
--- INTEGRACIÓN COMPLETADA ---
Dataset MASTER guardado con 5137 registros y 21 columnas.
